# SurfaceTarget-CRC — phase 04 — patch 4 3

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 4 — PATCH 4.3
#
# COLORECTAL COHORT REFINEMENT AND
# CRISPR DEPENDENCY ROBUSTNESS
#
# Objectives:
# 1. Validate existing DepMap inputs
# 2. Correct colorectal adenocarcinoma cohort definition
# 3. Exclude benign adenoma from primary CRC analysis
# 4. Audit repeated originating patient identifiers
# 5. Recalculate candidate CRISPR dependency summaries
# 6. Calculate one-model-per-patient sensitivity estimates
# 7. Account for gene-level missingness
# 8. Review other cancer lineage measurements
# 9. Generate descriptive dependency figure
# 10. Save corrected results and provenance
#
# IMPORTANT:
# - No MSS status inferred
# - No permanent exclusion of research candidates
# - No therapeutic target scores assigned
# - No modification of original DepMap datasets
# - No changes to Patch 4.2 outputs
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import hashlib
import json
import math


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"

DEPMAP = ROOT / "data/raw/depmap/26Q3"

RESULTS = ROOT / "results/tables"

FIGURES = ROOT / "results/figures"

for directory in [META, RESULTS, FIGURES]:
    directory.mkdir(parents=True, exist_ok=True)


MODEL_PATH = DEPMAP / "Model.csv"

CRISPR_PATH = DEPMAP / "CRISPRGeneEffect.csv"

IMPORT_PATH = (
    META / "depmap_26q3_import_verification.csv"
)

MAPPING_PATH = (
    RESULTS / "depmap_26q3_candidate_gene_mapping.csv"
)


LEADING_GENES = [
    "CDH3",
    "CLDN1",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "CTHRC1",
    "S100P",
    "KRT80",
    "CEMIP",
    "INHBA",
    "WNT2"
]


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 4 — PATCH 4.3")
print("CRC DEPENDENCY ROBUSTNESS AUDIT")
print("=" * 65)


# ------------------------------------------------------------
# 2. VALIDATE INPUTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING DEPMAP INPUTS")
print("-" * 65)


for path in [
    MODEL_PATH,
    CRISPR_PATH,
    IMPORT_PATH,
    MAPPING_PATH
]:

    if not path.exists():
        raise FileNotFoundError(path)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


audit = pd.read_csv(
    IMPORT_PATH,
    dtype=str,
    keep_default_na=False
)

audit = audit.set_index("filename")


for path in [MODEL_PATH, CRISPR_PATH]:

    if path.name not in audit.index:
        raise RuntimeError(
            f"Missing checksum record: {path.name}"
        )

    expected = audit.loc[
        path.name,
        "sha256"
    ]

    actual = sha256_file(path)

    if expected != actual:
        raise RuntimeError(
            f"Source checksum mismatch: {path.name}"
        )


print("\nLocally recorded source checksums: PASSED")

print(
    "Official release checksum comparison "
    "remains unavailable."
)


# ------------------------------------------------------------
# 3. LOAD MODEL METADATA
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING SCREENED CANCER MODELS")
print("-" * 65)


models = pd.read_csv(
    MODEL_PATH,
    dtype=str,
    keep_default_na=False
)


if models["ModelID"].duplicated().any():
    raise RuntimeError(
        "Duplicate DepMap model IDs."
    )


models["ModelID"] = (
    models["ModelID"]
    .astype(str)
    .str.strip()
)


model_columns = [
    "ModelID",
    "PatientID",
    "CellLineName",
    "OncotreeLineage",
    "OncotreePrimaryDisease",
    "OncotreeSubtype"
]


if not set(model_columns).issubset(models.columns):
    raise RuntimeError(
        "Required model metadata columns missing."
    )


# ------------------------------------------------------------
# 4. LOAD CANDIDATE GENE MAPPING
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING PREVIOUS GENE MAPPINGS")
print("-" * 65)


mapping = pd.read_csv(
    MAPPING_PATH,
    dtype=str,
    keep_default_na=False
)


if len(mapping) != 948:
    raise RuntimeError(
        "Expected 948 candidate genes."
    )


valid_mapping = mapping[
    mapping["depmap_mapping_status"].eq(
        "Unique exact gene-symbol match"
    )
].copy()


if len(valid_mapping) != 930:
    raise RuntimeError(
        "Expected 930 previously matched genes."
    )


if valid_mapping["depmap_column"].duplicated().any():
    raise RuntimeError(
        "Duplicate selected DepMap columns."
    )


print("\nCandidates:", len(mapping))

print("Unique exact symbol matches:", len(valid_mapping))

print("Unresolved mappings:", len(mapping) - len(valid_mapping))


# ------------------------------------------------------------
# 5. EXTRACT CANDIDATE CRISPR MATRIX
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING CANDIDATE CRISPR MEASUREMENTS")
print("-" * 65)


header = pd.read_csv(
    CRISPR_PATH,
    nrows=0
).columns.tolist()


if len(header) != 18435:
    raise RuntimeError(
        "Unexpected CRISPR gene-column count."
    )


model_column = header[0]


if model_column not in [
    "Unnamed: 0",
    "ModelID"
]:
    raise RuntimeError(
        "Unexpected CRISPR model-ID column."
    )


selected_columns = (
    valid_mapping["depmap_column"]
    .tolist()
)


if not set(selected_columns).issubset(set(header)):
    raise RuntimeError(
        "Candidate columns missing from CRISPR file."
    )


effects = pd.read_csv(
    CRISPR_PATH,
    usecols=[model_column] + selected_columns,
    dtype={model_column: str},
    low_memory=False
)


effects = effects.rename(
    columns={model_column: "ModelID"}
)


effects["ModelID"] = (
    effects["ModelID"]
    .astype(str)
    .str.strip()
)


if effects["ModelID"].duplicated().any():
    raise RuntimeError(
        "Duplicate screened ModelIDs."
    )


effects = effects.set_index("ModelID")


effects = effects.apply(
    pd.to_numeric,
    errors="raise"
)


if np.isinf(
    effects.to_numpy(dtype=float)
).any():
    raise RuntimeError(
        "Infinite CRISPR gene-effect values."
    )


if effects.shape != (1361, 930):
    raise RuntimeError(
        f"Unexpected candidate matrix: {effects.shape}"
    )


print("\nCandidate CRISPR matrix:", effects.shape)

print("Candidate matrix validation: PASSED")


# ------------------------------------------------------------
# 6. ALIGN SCREENED MODEL METADATA
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("ALIGNING SCREENED MODEL METADATA")
print("-" * 65)


model_indexed = models.set_index("ModelID")


if not set(effects.index).issubset(
    set(model_indexed.index)
):
    raise RuntimeError(
        "Screened models missing metadata."
    )


screened = model_indexed.loc[
    effects.index
].copy()


screened.index.name = "ModelID"


print("\nScreened models:", len(screened))

print("Metadata correspondence: PASSED")


# ------------------------------------------------------------
# 7. CORRECT CRC ADENOCARCINOMA DEFINITION
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("REFINING COLORECTAL CANCER COHORT")
print("-" * 65)


# The previous patch included one model
# classified as Tubular Adenoma of the Colon.
#
# The primary cohort must explicitly carry
# the colorectal adenocarcinoma disease label.


crc_mask = (
    screened["OncotreePrimaryDisease"]
    .str.strip()
    .eq("Colorectal Adenocarcinoma")
)


crc_models = screened[
    crc_mask
].copy()


adenoma_mask = (
    screened["OncotreePrimaryDisease"]
    .eq("Tubular Adenoma of the Colon")
)


adenoma_models = screened[
    adenoma_mask
].copy()


print("\nColorectal adenocarcinoma models:",
      len(crc_models))

print("Tubular colon adenoma models:",
      len(adenoma_models))


if len(crc_models) != 81:
    raise RuntimeError(
        "Expected 81 colorectal adenocarcinoma models."
    )


if len(adenoma_models) != 1:
    raise RuntimeError(
        "Expected one tubular adenoma model."
    )


print("\nCRC subtype distribution:")

print(
    crc_models[
        "OncotreeSubtype"
    ].value_counts().to_string()
)


print("\nExcluded precursor model:")

print(
    adenoma_models[
        [
            "CellLineName",
            "OncotreePrimaryDisease",
            "OncotreeSubtype"
        ]
    ].to_string()
)


print("\nPrimary CRC cohort refinement: PASSED")


# ------------------------------------------------------------
# 8. DEFINE OTHER-LINEAGE COMPARATOR
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DEFINING OTHER-LINEAGE COMPARATOR")
print("-" * 65)


# Exclude Bowel-lineage models from the
# primary comparator.
#
# This prevents benign colorectal precursor
# models and other bowel diseases from entering
# the comparator group.
#
# Also exclude models explicitly annotated
# as benign/non-cancerous elsewhere.
#
# This is a descriptive comparator, not
# a clinically matched control cohort.


bowel_mask = (
    screened["OncotreeLineage"]
    .str.contains(
        r"bowel",
        case=False,
        regex=True,
        na=False
    )
)


nonmalignant_mask = (
    screened["OncotreePrimaryDisease"]
    .str.contains(
        r"non[- ]?cancerous|"
        r"benign|adenoma|"
        r"normal tissue|"
        r"hyperplasia|"
        r"dysplasia|"
        r"sessile serrated",
        case=False,
        regex=True,
        na=False
    )
)


other_mask = (
    ~bowel_mask
    & ~nonmalignant_mask
    & ~crc_mask
)


other_models = screened[
    other_mask
].copy()


print("\nCRC adenocarcinoma models:",
      len(crc_models))

print(
    "Other-lineage comparator models:",
    len(other_models)
)

print(
    "Screened bowel-lineage models:",
    int(bowel_mask.sum())
)

print(
    "Explicit nonmalignant/precursor "
    "annotations:",
    int(nonmalignant_mask.sum())
)


if len(other_models) < 500:
    raise RuntimeError(
        "Unexpectedly small other-lineage comparator."
    )


print("\nComparator definition: PASSED")

print(
    "Other-lineage models are heterogeneous "
    "and are not independent biological controls."
)


# ------------------------------------------------------------
# 9. REPEATED PATIENT AUDIT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("ORIGINATING PATIENT DUPLICATION AUDIT")
print("-" * 65)


crc_patient = crc_models.reset_index().copy()


crc_patient["PatientID"] = (
    crc_patient["PatientID"]
    .astype(str)
    .str.strip()
)


# Missing PatientIDs are treated as separate
# unknown-origin models, rather than incorrectly
# grouping all missing values as one patient.


crc_patient["patient_group"] = np.where(
    crc_patient["PatientID"].ne(""),
    crc_patient["PatientID"],
    "UNKNOWN_" + crc_patient["ModelID"]
)


duplicate_patients = crc_patient[
    crc_patient["patient_group"].duplicated(
        keep=False
    )
].copy()


print("\nModels sharing a recorded patient ID:",
      len(duplicate_patients))


if not duplicate_patients.empty:

    print("\nRepeated-patient models:")

    print(
        duplicate_patients[
            [
                "PatientID",
                "ModelID",
                "CellLineName",
                "OncotreeSubtype"
            ]
        ]
        .sort_values(
            ["PatientID", "ModelID"]
        )
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 10. ONE-MODEL-PER-PATIENT SENSITIVITY COHORT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CONSTRUCTING PATIENT-DEDUPLICATED COHORT")
print("-" * 65)


crc_patient["measured_candidate_genes"] = (
    effects.loc[
        crc_patient["ModelID"]
    ]
    .notna()
    .sum(axis=1)
    .to_numpy()
)


# Deterministic sensitivity selection:
#
# 1. Keep model with most measured candidate genes.
# 2. Break ties by ModelID.
#
# This is a robustness assessment, not
# a biological claim that one model is better.


crc_patient_sorted = crc_patient.sort_values(
    [
        "measured_candidate_genes",
        "ModelID"
    ],
    ascending=[
        False,
        True
    ]
)


deduplicated = (
    crc_patient_sorted
    .drop_duplicates(
        subset="patient_group",
        keep="first"
    )
    .copy()
)


deduplicated_ids = (
    deduplicated["ModelID"]
    .tolist()
)


print("\nOriginal CRC models:", len(crc_models))

print(
    "One-model-per-patient sensitivity models:",
    len(deduplicated_ids)
)


if len(deduplicated_ids) > len(crc_models):
    raise RuntimeError(
        "Sensitivity cohort unexpectedly expanded."
    )


print("Patient deduplication: PASSED")


# ------------------------------------------------------------
# 11. BUILD ANALYSIS MATRICES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING CRISPR ANALYSIS GROUPS")
print("-" * 65)


crc_effects = effects.loc[
    crc_models.index
].copy()


other_effects = effects.loc[
    other_models.index
].copy()


sensitivity_effects = effects.loc[
    deduplicated_ids
].copy()


print("\nCRC matrix:", crc_effects.shape)

print("Other-lineage matrix:", other_effects.shape)

print(
    "Patient-deduplicated matrix:",
    sensitivity_effects.shape
)


# ------------------------------------------------------------
# 12. GENE-EFFECT SUMMARY FUNCTION
# ------------------------------------------------------------

def summarize_effect(values):

    values = np.asarray(
        values,
        dtype=float
    )

    values = values[
        np.isfinite(values)
    ]

    n = len(values)

    if n == 0:

        return {
            "n": 0,
            "median": np.nan,
            "q25": np.nan,
            "q75": np.nan,
            "fraction_lt_05": np.nan,
            "fraction_lt_1": np.nan
        }

    return {

        "n": int(n),

        "median": float(
            np.median(values)
        ),

        "q25": float(
            np.percentile(values, 25)
        ),

        "q75": float(
            np.percentile(values, 75)
        ),

        "fraction_lt_05": float(
            np.mean(values < -0.5)
        ),

        "fraction_lt_1": float(
            np.mean(values < -1.0)
        )

    }


# ------------------------------------------------------------
# 13. RECALCULATE CANDIDATE DEPENDENCIES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CALCULATING CORRECTED CRISPR DEPENDENCIES")
print("-" * 65)


records = []


for _, row in mapping.iterrows():

    gene_name = row["gene_name"]

    depmap_col = row["depmap_column"]

    result = row.to_dict()

    if (
        not depmap_col
        or depmap_col not in effects.columns
    ):

        crc_stats = summarize_effect([])
        other_stats = summarize_effect([])
        sensitivity_stats = summarize_effect([])

    else:

        crc_stats = summarize_effect(
            crc_effects[depmap_col]
        )

        other_stats = summarize_effect(
            other_effects[depmap_col]
        )

        sensitivity_stats = summarize_effect(
            sensitivity_effects[depmap_col]
        )


    crc_med = crc_stats["median"]

    other_med = other_stats["median"]

    sensitivity_med = sensitivity_stats["median"]


    result.update({

        "crc_models_total":
            len(crc_effects),

        "crc_models_measured":
            crc_stats["n"],

        "crc_measurement_fraction":
            crc_stats["n"] / len(crc_effects),

        "crc_median_gene_effect":
            crc_med,

        "crc_q25_gene_effect":
            crc_stats["q25"],

        "crc_q75_gene_effect":
            crc_stats["q75"],

        "crc_fraction_below_minus_05":
            crc_stats["fraction_lt_05"],

        "crc_fraction_below_minus_1":
            crc_stats["fraction_lt_1"],

        "other_models_measured":
            other_stats["n"],

        "other_median_gene_effect":
            other_med,

        "other_fraction_below_minus_05":
            other_stats["fraction_lt_05"],

        "crc_minus_other_median_effect":
            (
                crc_med - other_med
                if np.isfinite(crc_med)
                and np.isfinite(other_med)
                else np.nan
            ),

        "deduplicated_models_measured":
            sensitivity_stats["n"],

        "deduplicated_median_gene_effect":
            sensitivity_med,

        "dedup_minus_primary_median":
            (
                sensitivity_med - crc_med
                if np.isfinite(sensitivity_med)
                and np.isfinite(crc_med)
                else np.nan
            )

    })

    records.append(result)


summary = pd.DataFrame(records)


if len(summary) != 948:
    raise RuntimeError(
        "Candidate count changed."
    )


print("\nCandidates summarized:", len(summary))


print(
    "Candidates with CRC measurements:",
    int(
        summary["crc_models_measured"]
        .gt(0)
        .sum()
    )
)


# ------------------------------------------------------------
# 14. COVERAGE-AWARE DEPENDENCY REVIEW
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENE COVERAGE AND DEPENDENCY REVIEW")
print("-" * 65)


# An 80% coverage threshold is used for
# descriptive robustness screening.
#
# It is not a biological dependency threshold.


MIN_CRC_COVERAGE = math.ceil(
    0.80 * len(crc_effects)
)


summary["high_crc_coverage"] = (
    summary["crc_models_measured"]
    .ge(MIN_CRC_COVERAGE)
)


print("\nCRC models:", len(crc_effects))

print(
    "Minimum for high coverage:",
    MIN_CRC_COVERAGE
)


print(
    "Candidates meeting high coverage:",
    int(
        summary["high_crc_coverage"].sum()
    )
)


low_coverage = summary[
    summary["crc_models_measured"].gt(0)
    & ~summary["high_crc_coverage"]
].copy()


print(
    "Mapped candidates with lower coverage:",
    len(low_coverage)
)


high_coverage_ranked = summary[
    summary["high_crc_coverage"]
].sort_values(
    "crc_median_gene_effect"
)


print("\nTop 20 negative effects among "
      "high-coverage candidates:")


print(
    high_coverage_ranked[
        [
            "gene_name",
            "crc_models_measured",
            "crc_median_gene_effect",
            "crc_fraction_below_minus_05",
            "other_median_gene_effect",
            "crc_minus_other_median_effect"
        ]
    ]
    .head(20)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ------------------------------------------------------------
# 15. LEADING CANDIDATE ROBUSTNESS REVIEW
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING CANDIDATE DEPENDENCY ROBUSTNESS")
print("-" * 65)


leading = summary[
    summary["gene_name"].isin(
        LEADING_GENES
    )
].copy()


leading["display_order"] = (
    leading["gene_name"].map({
        gene: idx
        for idx, gene in enumerate(
            LEADING_GENES
        )
    })
)


leading = leading.sort_values(
    "display_order"
)


print("\n")


print(
    leading[
        [
            "gene_name",
            "crc_models_measured",
            "crc_median_gene_effect",
            "crc_fraction_below_minus_05",
            "other_median_gene_effect",
            "deduplicated_median_gene_effect",
            "dedup_minus_primary_median"
        ]
    ]
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ------------------------------------------------------------
# 16. CROSS-LINEAGE DESCRIPTIVE AUDIT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CROSS-LINEAGE DEPENDENCY COVERAGE")
print("-" * 65)


lineage_counts = (
    other_models["OncotreeLineage"]
    .replace("", "NOT AVAILABLE")
    .value_counts()
)


print("\nOther-lineage model distribution:")

print(
    lineage_counts.to_string()
)


# Save descriptive summaries for the
# biologically relevant leading genes.
#
# No inferential lineage-specificity
# claims are made.


lineage_records = []


for gene in LEADING_GENES:

    matching = valid_mapping[
        valid_mapping["gene_name"].eq(gene)
    ]

    if len(matching) != 1:
        continue

    column = matching.iloc[0]["depmap_column"]

    for lineage, group in other_models.groupby(
        "OncotreeLineage"
    ):

        values = effects.loc[
            group.index,
            column
        ]

        stats = summarize_effect(values)

        lineage_records.append({

            "gene_name": gene,

            "lineage": lineage,

            "models_total": len(group),

            "models_measured": stats["n"],

            "median_gene_effect": stats["median"],

            "fraction_below_minus_05":
                stats["fraction_lt_05"]

        })


lineage_summary = pd.DataFrame(
    lineage_records
)


print(
    "\nLeading-gene lineage summaries:",
    len(lineage_summary)
)


print(
    "These summaries are descriptive; "
    "differences may reflect lineage composition."
)


# ------------------------------------------------------------
# 17. GENERATE LEADING-GENE DEPENDENCY FIGURE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING DEPENDENCY COMPARISON FIGURE")
print("-" * 65)


plot_data = leading.dropna(
    subset=[
        "crc_median_gene_effect",
        "other_median_gene_effect"
    ]
).copy()


positions = np.arange(
    len(plot_data)
)


fig, ax = plt.subplots(
    figsize=(11, 7)
)


ax.scatter(
    plot_data["crc_median_gene_effect"],
    positions,
    s=70,
    label="CRC adenocarcinoma"
)


ax.scatter(
    plot_data["other_median_gene_effect"],
    positions,
    s=70,
    marker="x",
    label="Other screened lineages"
)


for position, (_, row) in enumerate(
    plot_data.iterrows()
):

    ax.plot(
        [
            row["crc_median_gene_effect"],
            row["other_median_gene_effect"]
        ],
        [position, position],
        alpha=0.35
    )


ax.axvline(
    0,
    linestyle="--",
    linewidth=1
)


ax.axvline(
    -0.5,
    linestyle=":",
    linewidth=1
)


ax.set_yticks(positions)


ax.set_yticklabels(
    plot_data["gene_name"]
)


ax.invert_yaxis()


ax.set_xlabel(
    "Median CRISPR gene-effect score"
)


ax.set_title(
    "DepMap 26Q3 — Leading SurfaceTarget-CRC Candidates"
)


ax.legend()


ax.grid(
    axis="x",
    alpha=0.2
)


fig.tight_layout()


figure_path = (
    FIGURES /
    "depmap_26q3_crc_dependency_comparison.png"
)


fig.savefig(
    figure_path,
    dpi=200,
    bbox_inches="tight"
)


plt.show()
plt.close(fig)


print("\nDependency comparison figure saved.")


# ------------------------------------------------------------
# 18. SAVE CORRECTED ANALYSIS OUTPUTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING CORRECTED DEPMAP RESULTS")
print("-" * 65)


crc_models.reset_index().to_csv(
    RESULTS /
    "depmap_26q3_refined_crc_models.csv",
    index=False
)


adenoma_models.reset_index().to_csv(
    RESULTS /
    "depmap_26q3_excluded_precursor_models.csv",
    index=False
)


duplicate_patients.to_csv(
    RESULTS /
    "depmap_26q3_repeated_patient_review.csv",
    index=False
)


deduplicated.to_csv(
    RESULTS /
    "depmap_26q3_patient_deduplicated_models.csv",
    index=False
)


summary.to_csv(
    RESULTS /
    "depmap_26q3_refined_dependency_summary.csv",
    index=False
)


low_coverage.to_csv(
    RESULTS /
    "depmap_26q3_low_coverage_gene_review.csv",
    index=False
)


leading.to_csv(
    RESULTS /
    "depmap_26q3_refined_leading_dependency.csv",
    index=False
)


lineage_summary.to_csv(
    RESULTS /
    "depmap_26q3_leading_gene_lineage_summary.csv",
    index=False
)


# Save corrected colorectal candidate matrix.
#
# Only the refined 81-model cohort is exported.

column_to_gene = dict(
    zip(
        valid_mapping["depmap_column"],
        valid_mapping["gene_name"]
    )
)


crc_export = crc_effects.rename(
    columns=column_to_gene
)


crc_export.index.name = "ModelID"


crc_export.to_csv(
    RESULTS /
    "depmap_26q3_refined_crc_candidate_matrix.csv"
)


print("\nAll refined analysis tables saved.")


# ------------------------------------------------------------
# 19. SAVE PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING RESEARCH PROVENANCE")
print("-" * 65)


provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "4.3",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "depmap_release": "Public 26Q3",

    "model_source_sha256":
        sha256_file(MODEL_PATH),

    "crispr_source_sha256":
        sha256_file(CRISPR_PATH),

    "official_source_checksum_verified": False,

    "original_crc_related_models": 82,

    "refined_crc_adenocarcinoma_models":
        int(len(crc_models)),

    "excluded_tubular_adenoma_models":
        int(len(adenoma_models)),

    "crc_primary_disease_rule": (
        "Exact match: Colorectal Adenocarcinoma"
    ),

    "other_lineage_comparator_models":
        int(len(other_models)),

    "comparator_definition": (
        "Screened non-Bowel models, excluding "
        "explicit benign and precursor annotations"
    ),

    "patient_deduplicated_crc_models":
        int(len(deduplicated_ids)),

    "patient_deduplication_rule": (
        "One model per nonmissing PatientID, "
        "selected by candidate gene coverage "
        "then lexicographic ModelID"
    ),

    "candidate_genes": 948,

    "exact_symbol_mapped_genes":
        int(len(valid_mapping)),

    "unresolved_gene_mappings":
        int(len(mapping) - len(valid_mapping)),

    "high_coverage_definition": (
        "Measurements in at least 80% "
        "of refined CRC adenocarcinoma models"
    ),

    "high_coverage_minimum_models":
        int(MIN_CRC_COVERAGE),

    "gene_effect_summary_statistics": [
        "Median",
        "25th percentile",
        "75th percentile",
        "Fraction below -0.5",
        "Fraction below -1.0",
        "Other-lineage median",
        "Patient-deduplicated CRC median"
    ],

    "msi_status_inferred": False,

    "formal_lineage_selectivity_testing": False,

    "therapeutic_target_ranking": False,

    "original_patch_42_outputs_overwritten": False,

    "limitations": [

        "Source checksum comparison with "
        "official DepMap release unavailable.",

        "Colorectal models are not confirmed MSS.",

        "Some models share originating patients.",

        "Initial gene matching uses exact "
        "gene-symbol correspondence.",

        "Gene effects are cancer-model "
        "fitness measurements.",

        "Other-lineage comparator is heterogeneous.",

        "Colorectal versus other-lineage median "
        "differences are descriptive.",

        "Low CRISPR dependency does not "
        "exclude antibody-targeting relevance.",

        "Stromal targets are not adequately "
        "modeled by cancer-cell-intrinsic screens.",

        "No therapeutically validated "
        "targets have been established."

    ]
}


with open(
    META /
    "depmap_26q3_refined_dependency_provenance.json",
    "w"
) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )


print("\nProvenance saved.")


# ------------------------------------------------------------
# 20. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 4.3 SUMMARY")
print("=" * 65)


print("\nDepMap release: Public 26Q3")


print(
    "Original colorectal-related models: 82"
)


print(
    "Refined CRC adenocarcinoma models:",
    len(crc_models)
)


print(
    "Tubular adenoma models withheld:",
    len(adenoma_models)
)


print(
    "Patient-deduplicated sensitivity models:",
    len(deduplicated_ids)
)


print(
    "\nCandidate genes:",
    len(mapping)
)


print(
    "Mapped genes:",
    len(valid_mapping)
)


print(
    "Genes with >=80% CRC coverage:",
    int(
        summary["high_crc_coverage"].sum()
    )
)


print(
    "\nOther-lineage comparator models:",
    len(other_models)
)


print("\nOutputs:")

print("1. Refined CRC model inventory")

print("2. Tubular adenoma exclusion audit")

print("3. Repeated-patient review")

print("4. Patient-deduplicated model inventory")

print("5. Corrected candidate dependency summary")

print("6. Low-coverage gene review")

print("7. Leading-candidate robustness review")

print("8. Other-lineage dependency summary")

print("9. Corrected CRC candidate matrix")

print("10. Dependency comparison figure")

print("11. Research provenance")


print("\nPrimary CRC model correction: COMPLETE")

print("Patient-duplication sensitivity: COMPLETE")

print("Coverage-aware dependency analysis: COMPLETE")


print("\nNo MSS status inferred.")

print("No source measurements modified.")

print("No original results overwritten.")

print("No final target ranking performed.")


print("\nPATCH 4.3 COMPLETE")

print("=" * 65)